# 03 — IXI Main Experiment: Brain Age Gap vs Face Age Gap

**Primary research question**: Do brain age gap and face age gap (both derived from the same T1 MRI) correlate across subjects?

**Hypothesis**: If both biomarkers reflect a common aging process, their residuals from chronological age should be positively correlated (ρ > 0.3).

**Dataset**: IXI — ~580 subjects, ages 20–86, 3 sites (Guys / HH / IOP).

**Pipeline**:
1. Load IXI T1 scans + demographics (IXI.xls)
2. Render frontal face PNG from each T1
3. FaceAge inference → face_age_gap = face_age - chron_age
4. SynthSeg/SFCN → brain_age_gap = brain_age - chron_age
5. Spearman ρ between the two gaps + site stratification
6. Correlation matrix plot for paper Figure 1

In [ ]:
import sys
from pathlib import Path

# ── CONFIG ──────────────────────────────────────────────────────────────────
IXI_T1_DIR    = Path("../data/ixi/T1")       # directory with IXI-T1-*.nii.gz
IXI_META_XLS  = Path("../data/ixi/IXI.xls")
FACEAGE_ROOT  = Path("../vendor/FaceAge")
RENDERS_DIR   = Path("../results/ixi_renders")
OUT_DIR       = Path("../results/ixi_main")
SKIN_LEVEL    = 40
BYPASS_MTCNN  = True
DEVICE        = "cpu"   # 'cuda' on Colab GPU
# ────────────────────────────────────────────────────────────────────────────

RENDERS_DIR.mkdir(parents=True, exist_ok=True)
OUT_DIR.mkdir(parents=True, exist_ok=True)
sys.path.insert(0, str(Path(".").resolve().parent))

In [ ]:
# ── 1. Load demographics ─────────────────────────────────────────────────────
import pandas as pd
import numpy as np

from src.utils import load_ixi_metadata

meta = load_ixi_metadata(IXI_META_XLS)
print(f"IXI metadata: {len(meta)} subjects")
print(meta[["IXI_ID", "AGE", "SEX_ID", "SITE"]].describe())
meta.head()

In [ ]:
# ── 2. Match T1 files to metadata ───────────────────────────────────────────
import re

t1_files = sorted(IXI_T1_DIR.glob("IXI*-T1.nii.gz"))
print(f"T1 files found: {len(t1_files)}")

# Extract IXI_ID from filename: IXI002-Guys-0828-T1.nii.gz → 2
def parse_ixi_id(fname: str) -> int | None:
    m = re.match(r"IXI(\d+)-", Path(fname).name)
    return int(m.group(1)) if m else None

file_df = pd.DataFrame({
    "IXI_ID": [parse_ixi_id(str(f)) for f in t1_files],
    "t1_path": [str(f) for f in t1_files],
}).dropna(subset=["IXI_ID"])
file_df["IXI_ID"] = file_df["IXI_ID"].astype(int)

df = meta.merge(file_df, on="IXI_ID", how="inner")
print(f"Subjects with T1 + metadata: {len(df)}")
print(df["SITE"].value_counts())

In [ ]:
# ── 3. Render face PNGs (parallel via batch script or inline) ────────────────
from tqdm.notebook import tqdm
from src.render import render_face

df["render_png"] = ""
df["render_ok"] = False

for idx, row in tqdm(df.iterrows(), total=len(df), desc="Rendering faces"):
    out_png = RENDERS_DIR / f"IXI{row.IXI_ID:03d}.png"
    ok = out_png.exists() or render_face(row.t1_path, out_png, level=SKIN_LEVEL)
    df.at[idx, "render_png"] = str(out_png)
    df.at[idx, "render_ok"] = bool(ok)

n_ok = df["render_ok"].sum()
print(f"Renders OK: {n_ok} / {len(df)}  ({100*n_ok/len(df):.1f}%)")

In [ ]:
# ── 4. FaceAge inference ─────────────────────────────────────────────────────
from src.face_age import predict_age_batch

good = df[df["render_ok"]].copy()
pngs = [Path(p) for p in good["render_png"]]

fa_results = predict_age_batch(
    img_paths=pngs,
    faceage_root=FACEAGE_ROOT,
    bypass_mtcnn=BYPASS_MTCNN,
    device=DEVICE,
)

fa_df = pd.DataFrame([{"render_png": r["path"], "face_age": r["age"]} for r in fa_results])
good = good.merge(fa_df, on="render_png", how="left")
good["face_age_gap"] = good["face_age"] - good["AGE"]

print(f"Face age gap: mean={good['face_age_gap'].mean():.2f}  std={good['face_age_gap'].std():.2f}")

In [ ]:
# ── 5. Brain age (SynthSeg or load cached) ───────────────────────────────────
BRAIN_AGE_CSV = OUT_DIR / "ixi_brain_ages.csv"

if BRAIN_AGE_CSV.exists():
    ba_df = pd.read_csv(BRAIN_AGE_CSV)
    print(f"Loaded cached brain ages: {len(ba_df)} rows")
else:
    from src.brain_age import run_synthseg, synthseg_to_brain_age
    synthseg_dir = OUT_DIR / "synthseg_vols"
    ba_records = []
    for _, row in tqdm(good.iterrows(), total=len(good), desc="SynthSeg"):
        try:
            vol_csv = run_synthseg(row.t1_path, synthseg_dir)
            ba = synthseg_to_brain_age(vol_csv)
        except Exception as e:
            print(f"  ⚠ IXI{row.IXI_ID}: {e}")
            ba = np.nan
        ba_records.append({"IXI_ID": row.IXI_ID, "brain_age": ba})
    ba_df = pd.DataFrame(ba_records)
    ba_df.to_csv(BRAIN_AGE_CSV, index=False)

good = good.merge(ba_df, on="IXI_ID", how="left")
good["brain_age_gap"] = good["brain_age"] - good["AGE"]

In [ ]:
# ── 6. Primary analysis: Spearman ρ(brain_age_gap, face_age_gap) ─────────────
from scipy import stats
import matplotlib.pyplot as plt
import seaborn as sns

analysis = good.dropna(subset=["face_age_gap", "brain_age_gap"])
print(f"Analysis N = {len(analysis)}")

rho, pval = stats.spearmanr(analysis["brain_age_gap"], analysis["face_age_gap"])
print(f"Spearman ρ = {rho:.3f}  p = {pval:.4g}")

# Site-stratified
print("\nSite-stratified:")
for site, grp in analysis.groupby("SITE"):
    r, p = stats.spearmanr(grp["brain_age_gap"], grp["face_age_gap"])
    print(f"  {site:6s}  n={len(grp):3d}  ρ={r:.3f}  p={p:.4g}")

In [ ]:
# ── 7. Figure 1 candidate: Correlation scatter + site color ──────────────────
site_colors = {"Guys": "#4C72B0", "HH": "#DD8452", "IOP": "#55A868"}

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: scatter brain gap vs face gap
ax = axes[0]
for site, grp in analysis.groupby("SITE"):
    ax.scatter(grp["brain_age_gap"], grp["face_age_gap"],
               alpha=0.5, s=15, label=site, color=site_colors[site])
ax.axhline(0, color="gray", lw=0.5)
ax.axvline(0, color="gray", lw=0.5)
ax.set_xlabel("Brain age gap (years)", fontsize=11)
ax.set_ylabel("Face age gap (years)", fontsize=11)
ax.set_title(f"Spearman ρ = {rho:.3f}  (n={len(analysis)}, p={pval:.3g})", fontsize=11)
ax.legend(fontsize=9)

# Right: age distribution
ax = axes[1]
ax.hist(analysis["AGE"], bins=20, edgecolor="white", color="steelblue", alpha=0.7)
ax.set_xlabel("Chronological age (years)", fontsize=11)
ax.set_ylabel("Count", fontsize=11)
ax.set_title("IXI age distribution", fontsize=11)

plt.suptitle("IXI: Brain age gap vs Face age gap from same T1 MRI", fontsize=13)
plt.tight_layout()
plt.savefig(OUT_DIR / "03_ixi_brain_vs_face_gap.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# ── 8. Save full results table ───────────────────────────────────────────────
out_csv = OUT_DIR / "ixi_all_predictions.csv"
good.to_csv(out_csv, index=False)
print(f"Saved → {out_csv}")
good[["IXI_ID", "AGE", "SEX_ID", "SITE", "face_age", "face_age_gap", "brain_age", "brain_age_gap"]].describe()